# BirdCLEF+ 2026 — Inference & Submission

加载训练好的模型权重，对测试声景做推理，生成提交文件。

**流程**: 声景文件 → 12 个 5s 窗口 → Mel 频谱图 → 模型集成 → 后处理 → submission.csv

In [ ]:
import os, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast

import librosa
import timm
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

In [ ]:
# ── 配置 ──
IS_KAGGLE = Path('/kaggle/input').exists()
DATA_DIR = Path('/kaggle/input/birdclef-2026') if IS_KAGGLE else Path('../data/raw')
# 模型权重作为 Kaggle Dataset 挂载
MODEL_DIR = Path('/kaggle/input/birdclef2026-weights') if IS_KAGGLE else Path('../models')
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('../models')

SR = 32_000
CLIP_DURATION = 5.0
N_FFT = 1024
HOP_LENGTH = 320
N_MELS = 128
FMIN = 50
FMAX = 14000
NUM_CLASSES = 234

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── 加载元信息 ──
submission = pd.read_csv(DATA_DIR / 'sample_submission.csv', nrows=0)
SPECIES_COLS = [c for c in submission.columns if c != 'row_id']
taxonomy = pd.read_csv(DATA_DIR / 'taxonomy.csv')
TAX_MAP = dict(zip(taxonomy['primary_label'].astype(str), taxonomy['class_name']))
LABEL2IDX = {label: i for i, label in enumerate(SPECIES_COLS)}

print(f'Species: {len(SPECIES_COLS)}')

In [ ]:
# ── 工具函数 ──
def load_audio(path, sr=SR, duration=CLIP_DURATION, offset=0.0):
    audio, _ = librosa.load(path, sr=sr, offset=offset, duration=duration, mono=True)
    target_len = int(sr * duration)
    if len(audio) < target_len:
        audio = np.tile(audio, (target_len // len(audio)) + 1)[:target_len]
    return audio[:target_len]


def audio_to_melspec(audio):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)


def compute_insect_energy(audio):
    S = np.abs(librosa.stft(audio, n_fft=1024))
    freqs = librosa.fft_frequencies(sr=SR, n_fft=1024)
    mask = (freqs >= 4000) & (freqs <= 6000)
    return float(S[mask].sum()) / (float(S.sum()) + 1e-8)


def parse_filename(filename):
    """从文件名解析 hour"""
    parts = Path(filename).stem.split('_')
    if len(parts) >= 6:
        return int(parts[5][:2])
    return 12

In [ ]:
# ── 模型 ──
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.adaptive_avg_pool2d(
            x.clamp(min=self.eps).pow(self.p), 1
        ).pow(1.0 / self.p).squeeze(-1).squeeze(-1)


class BirdCLEFB0(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            'tf_efficientnet_b0_ns', pretrained=False, in_chans=1,
            num_classes=0, global_pool=''
        )
        self.gem = GeM(p=3.0)
        self.hour_embed = nn.Embedding(24, 32)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(1280, 512)
        self.fc2 = nn.Linear(512 + 33, NUM_CLASSES)

    def forward(self, mel, insect_energy=None, hour=None):
        feat = self.backbone(mel)
        feat = self.gem(feat)
        feat = self.dropout(feat)
        feat = F.relu(self.fc1(feat))
        B = feat.size(0)
        h = self.hour_embed(hour) if hour is not None else torch.zeros(B, 32, device=feat.device)
        ie = insect_energy if insect_energy is not None else torch.zeros(B, 1, device=feat.device)
        return self.fc2(torch.cat([feat, torch.cat([h, ie], -1)], -1))

In [ ]:
# ── 加载模型 ──
models = []
for p in sorted(MODEL_DIR.glob('best_fold*.pth')):
    m = BirdCLEFB0().to(DEVICE)
    m.load_state_dict(torch.load(p, map_location=DEVICE))
    m.eval()
    models.append(m)
print(f'Loaded {len(models)} models')

In [ ]:
# ── 后处理 ──
TIME_PRIOR = {
    'Aves':     [0.2,0.2,0.3,0.5,0.9,1.0,1.0,0.8,0.6,0.4,0.3,0.3,
                 0.3,0.3,0.3,0.4,0.5,0.8,1.0,0.6,0.3,0.2,0.2,0.2],
    'Amphibia': [0.8,0.8,0.7,0.5,0.3,0.2,0.2,0.1,0.1,0.1,0.1,0.1,
                 0.1,0.1,0.1,0.1,0.2,0.4,0.8,1.0,1.0,1.0,0.9,0.9],
    'Insecta':  [0.5,0.5,0.5,0.8,0.6,0.4,1.0,1.0,0.4,0.3,0.3,0.3,
                 0.3,0.3,0.3,0.3,0.3,0.4,0.5,0.8,0.5,0.5,0.5,0.5],
}

SONOTYPE_GROUPS = {
    'A': ['47158son08','47158son11','47158son20'],
    'B': ['47158son13','47158son22','47158son23'],
    'C': ['47158son15','47158son16','47158son25'],
    'D': ['47158son04','47158son10'],
}

# 构建共现矩阵
labels_df = pd.read_csv(DATA_DIR / 'train_soundscapes_labels.csv')
cond_prob = np.zeros((NUM_CLASSES, NUM_CLASSES))
for _, row in labels_df.iterrows():
    idxs = [LABEL2IDX[l.strip()] for l in str(row['primary_label']).split(';')
            if l.strip() in LABEL2IDX]
    for a in idxs:
        for b in idxs:
            if a != b:
                cond_prob[a][b] += 1
cond_prob /= (cond_prob.sum(axis=1, keepdims=True) + 1e-8)
print(f'Co-occurrence matrix: {(cond_prob > 0).sum()} non-zero entries')


def postprocess(preds, hour):
    p = preds.copy()
    # 共现
    detected = np.where(p > 0.5)[0]
    for a in detected:
        for b in range(len(p)):
            if cond_prob[a][b] > 0.3:
                p[b] = max(p[b], p[a] * cond_prob[a][b] * 0.15)
    # 时间
    h = hour % 24
    for idx, col in enumerate(SPECIES_COLS):
        cls = TAX_MAP.get(col, '')
        if cls in TIME_PRIOR:
            f = TIME_PRIOR[cls][h]
            p[idx] = p[idx] * 0.9 + p[idx] * f * 0.1
    # Sonotype 拆分
    col2idx = {c: i for i, c in enumerate(SPECIES_COLS)}
    for members in SONOTYPE_GROUPS.values():
        mi = [col2idx[m] for m in members if m in col2idx]
        if mi:
            mx = max(p[i] for i in mi)
            for i in mi:
                p[i] = mx / len(mi)
    return p

In [ ]:
# ── 推理 ──
test_dir = DATA_DIR / 'test_soundscapes'
test_files = sorted(test_dir.glob('*.ogg'))
print(f'Test files: {len(test_files)}')

rows = []
for filepath in tqdm(test_files, desc='Inference'):
    stem = filepath.stem
    hour = parse_filename(str(filepath))

    for start_sec in range(0, 60, 5):
        audio = load_audio(str(filepath), offset=float(start_sec))
        mel = audio_to_melspec(audio)
        mel_t = torch.from_numpy(mel).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
        ie_t = torch.tensor([[compute_insect_energy(audio)]], dtype=torch.float32, device=DEVICE)

        fold_preds = []
        for model in models:
            with torch.no_grad(), autocast():
                logits = model(mel_t, insect_energy=ie_t)
            fold_preds.append(torch.sigmoid(logits).cpu().numpy()[0])
        avg_pred = np.mean(fold_preds, axis=0)
        avg_pred = postprocess(avg_pred, hour)

        row_id = f'{stem}_{start_sec + 5}'
        row = {'row_id': row_id}
        for i, col in enumerate(SPECIES_COLS):
            row[col] = avg_pred[i]
        rows.append(row)

sub = pd.DataFrame(rows)
sub.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
print(f'Submission: {len(sub)} rows')
sub.head()